# Part B · NB5 — Final Comparison (Label-Efficiency vs Part A)  ·  `pavecrack1300-partb-comparison`

**Course:** CSE 438 — Digital Image Processing · EWU · Summer 2026 · **Group:** _`<fill>`_

Pulls each SSL method's `results_partB.json` **and** the Part-A `results.json`, then builds the **label-efficiency**
table + chart (Task E) and consolidated error analysis (Task F). Attach all four method outputs **and** the Part-A
NB1/NB2/NB3 output (for its `results.json`).

In [ ]:
import glob, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
WORK = Path("/kaggle/working")
# merge all Part-B method results
pb = {}
for rp in glob.glob("/kaggle/input/**/results_partB.json", recursive=True):
    try: pb.update(json.load(open(rp)))
    except Exception as e: print("skip", rp, e)
# Part-A supervised baseline (best model = DeepLabV3)
partA = {}
for rp in glob.glob("/kaggle/input/**/results.json", recursive=True):
    try: partA.update(json.load(open(rp)))
    except Exception: pass
baseline = max(partA.values(), key=lambda r: r.get("miou", 0)) if partA else None
print("Part-B methods:", list(pb.keys()))
print("Part-A baseline:", baseline["model"] if baseline else "NOT FOUND — attach Part-A output")

## Task E — label-efficiency comparison table

**How to read this table honestly — two caveats that must appear in the report (§6.3, §6.8):**

1. **Label budget (the point of the experiment).** Every SSL row was fine-tuned on the **Part-A validation split
   (195 labelled images)**; the Part-A supervised baseline was trained on the **full Part-A train split
   (909 labelled images)**. So the SSL rows use **~21%** of the labels.
2. **Input resolution differs.** Part A trained/evaluated at **512×512**; Part B runs at **224×224** — the one
   resolution that every SSL encoder accepts (ResNet, and ViT-S/**14** → 16×16 tokens) and that fits the Kaggle
   T4 for multi-crop pretraining. Cracks are thin structures, so lower resolution costs IoU independently of the
   label budget. **Part of the Part-A → Part-B gap is therefore resolution, not only supervision.** We report the
   comparison as the assignment requires, but we do **not** claim the whole gap is a label-efficiency effect.


3. **The test split serves a double role.** Per the assignment's own simplification, the Part-A **test** split was
   used *both* to monitor fine-tuning / select the best checkpoint **and** to report these final metrics. It was
   therefore not a fully untouched hold-out, which **inflates** every Part-B number somewhat. The strictly correct
   fix would be a fourth, never-touched slice — not affordable here because the labelled set is only 195 images.
   State this explicitly in report §6.3.

*(The failure-overlap analysis further below is unaffected by this: it compares per-image **rankings** on the same
images, not absolute IoU values.)*

In [ ]:
rows = []
N_LAB = next(iter(pb.values()))["n_labelled"] if pb else 195
for name in sorted(pb, key=lambda k: -pb[k]["miou"]):
    r = pb[name]
    rows.append(dict(Method=name, Backbone="ResNet-50" if name!="DINOv2" else "ViT-S/14",
                     Labels=r["n_labelled"], mIoU=round(r["miou"],4), CrackIoU=round(r["crack_iou"],4),
                     Dice=round(r["mdice"],4), Params_M=r["params_M"]))
if baseline:
    rows.append(dict(Method="Part-A supervised (DeepLabV3)", Backbone="ResNet-50",
                     Labels=909, mIoU=round(baseline["miou"],4), CrackIoU=round(baseline.get("crack_iou",baseline["per_class_iou"][1]),4),
                     Dice=round(baseline.get("mdice",float('nan')),4), Params_M=baseline["params_M"]))
tbl = pd.DataFrame(rows).set_index("Method")
if baseline:
    base_miou = baseline["miou"]
    tbl["%_of_supervised"] = (100*tbl["mIoU"]/base_miou).round(1)
tbl.to_csv(WORK/"partB_label_efficiency.csv")
print(f"Label budget : SSL fine-tuned on {N_LAB} labelled images vs 909 for the Part-A baseline "
      f"({100*N_LAB/909:.0f}% of the labels)")
print("Resolution   : Part-B runs at 224x224, Part-A at 512x512 -> part of the gap is resolution, "
      "not only supervision (state this in report Sections 6.3/6.8).")
tbl

## Task E — chart: SSL mIoU vs the supervised baseline

In [ ]:
order = [m for m in ["SimCLR","BYOL","MAE","DINOv2"] if m in pb]
mious = [pb[m]["miou"] for m in order]
fig, ax = plt.subplots(figsize=(8,4.5))
b = ax.bar(order, mious, color=["#4C72B0","#DD8452","#55A868","#C44E52"][:len(order)])
ax.bar_label(b, fmt="%.3f")
if baseline:
    ax.axhline(baseline["miou"], ls="--", color="k", label=f"Part-A supervised ({baseline['miou']:.3f}, 909 labels @512px)")
ax.set_ylabel("test mIoU"); ax.set_ylim(0,1); ax.legend()
ax.set_title(f"Label-efficiency: SSL on {N_LAB} labels @224px vs supervised on 909 @512px")
plt.tight_layout(); plt.savefig(WORK/"partB_label_efficiency.png", dpi=120, bbox_inches="tight"); plt.show()

## Task F — consolidated error analysis & label-efficiency discussion
Best SSL method (highest test mIoU) recovers the largest share of the fully-supervised performance while using
only ~21% of the labels. Per-image crack-IoU (stored per method) lets us check whether the SSL models fail on the
same images as the Part-A supervised models — see the report's Insights section.

In [ ]:
if pb:
    best = max(pb, key=lambda k: pb[k]["miou"])
    print(f"Best SSL method: {best} (mIoU {pb[best]['miou']:.4f})")
    if baseline:
        print(f"= {100*pb[best]['miou']/baseline['miou']:.1f}% of the Part-A supervised mIoU, using {N_LAB}/909 = {100*N_LAB/909:.0f}% of the labels")
    print("\nRanking by label-efficiency (test mIoU):")
    for m in sorted(pb, key=lambda k:-pb[k]['miou']): print(f"  {m:10s} mIoU {pb[m]['miou']:.4f} | crackIoU {pb[m]['crack_iou']:.4f}")

## Task F — do the SSL models fail on the *same* images as the Part-A supervised models?
Both Part A and Part B stored **per-image crack IoU** on the identical test split, so we can compare failure modes
directly rather than by eye: we correlate each SSL method's per-image IoU with the Part-A baseline's, and measure
how many of the 20 hardest images they share.

In [ ]:
baseA = baseline.get("per_image_crack_iou") if baseline else None
if not baseA:
    print("Part-A per-image IoUs not found — attach the Part-A NB1 output to run this comparison.")
else:
    import pandas as pd, numpy as np
    rows = []
    worst_A = set(pd.Series(baseA).nsmallest(20).index)
    for m in order:
        pi = pb[m].get("per_image_crack_iou", {})
        common = sorted(set(pi) & set(baseA))
        if not common: continue
        a = np.array([baseA[s] for s in common]); b = np.array([pi[s] for s in common])
        ok = ~(np.isnan(a) | np.isnan(b))
        corr = float(np.corrcoef(a[ok], b[ok])[0, 1]) if ok.sum() > 2 else float("nan")
        worst_B = set(pd.Series(pi).nsmallest(20).index)
        rows.append(dict(Method=m, corr_with_PartA=round(corr, 3),
                         shared_worst20=len(worst_A & worst_B),
                         mean_IoU_on_PartA_worst20=round(float(np.nanmean([pi.get(s, np.nan) for s in worst_A])), 3)))
    cmp_df = pd.DataFrame(rows).set_index("Method")
    cmp_df.to_csv(WORK/"partB_failure_overlap.csv")
    print("Per-image crack-IoU agreement with the Part-A supervised baseline:\n")
    print(cmp_df.to_string())
    print("\nHigh correlation + large shared-worst overlap => the SSL models inherit the SAME hard images as the")
    print("supervised baseline (difficulty is data-intrinsic). Low overlap => the SSL encoder fails differently,")
    print("i.e. its pretext task transferred different texture/shape priors. Discuss in report Section 6.7.")

**NB5 complete.** Saved the label-efficiency table + chart. These feed report §6.6 (Results) and §6.8 (Label-Efficiency).